In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

warnings.filterwarnings('ignore')

# 1. Load Data
df = pd.read_csv('Heart.csv')

# 2. Data Preprocessing
# Handle missing values represented as '?' or NaN
df = df.replace('?', np.nan)
df['Ca'] = pd.to_numeric(df['Ca'], errors='coerce').fillna(df['Ca'].mode()[0])
df['Thal'] = df['Thal'].fillna(df['Thal'].mode()[0])

# Define features and target
X = df.drop(columns=['AHD', 'HD'])
y = df['HD'] # Target variable (0 or 1)

# Identify categorical and numeric columns
categorical_cols = ['Sex', 'ChestPain', 'RestECG', 'Slope', 'Thal']
numeric_cols = ['Age', 'RestBP', 'Chol', 'Fbs', 'MaxHR', 'ExAng', 'Oldpeak', 'Ca']

# Create preprocessing pipelines
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Define and Train Models
models = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42, max_iter=1000))]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))]),
    'XGBoost': Pipeline([('preprocessor', preprocessor), ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))])
}

metrics_data = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    metrics = {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_pred_proba), 4)
    }
    metrics_data.append(metrics)
    
    # Save Model
    joblib.dump(model, f'model_{name.replace(" ", "_").lower()}.pkl')
    print(f"Saved model_{name.replace(' ', '_').lower()}.pkl")

# 5. Save Metrics and Preprocessor info
pd.DataFrame(metrics_data).to_csv('model_metrics.csv', index=False)
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(numeric_cols + list(model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols)), 'feature_names.pkl')

print("Training and saving complete! You can now run the Streamlit app.")

Training Logistic Regression...
Saved model_logistic_regression.pkl
Training Random Forest...
Saved model_random_forest.pkl
Training XGBoost...
Saved model_xgboost.pkl
Training and saving complete! You can now run the Streamlit app.
